# AeroRUL — Phase 1: C-MAPSS Dataset Understanding

**Objective**: Understand, validate, organize, and document the NASA C-MAPSS dataset for subsequent RUL prediction modeling.

**Dataset**: NASA C-MAPSS (Commercial Modular Aero-Propulsion System Simulation)

**Reference**: Saxena et al., *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation*, PHM08

---

## 1. Project Context

The goal of the AeroRUL project is to estimate the **Remaining Useful Life (RUL)** of turbofan aircraft engines from multivariate sensor degradation patterns.

The C-MAPSS dataset contains simulated run-to-failure data for turbofan engines under varying operating conditions and fault modes:

| Subset | Operating Conditions | Fault Modes |
|--------|---------------------|-------------|
| FD001  | 1 (Sea Level)       | 1 (HPC)     |
| FD002  | 6                   | 1 (HPC)     |
| FD003  | 1 (Sea Level)       | 2 (HPC+Fan) |
| FD004  | 6                   | 2 (HPC+Fan) |

**Phase 1** focuses exclusively on data understanding — no modeling is performed.

## 2. Imports

In [4]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    AVAILABLE_SUBSETS, COLUMN_NAMES, SENSOR_COLUMNS,
    OP_SETTING_COLUMNS, SUBSET_METADATA, FIGURES_DIR
)
from src.data_loader import discover_datasets, load_cmapss, load_all_subsets
from src.data_validation import (
    validate_structure, validate_missing_values, validate_engines,
    get_engine_final_cycles, validate_sensors, analyze_operating_conditions,
    sensor_cycle_correlation, sensor_correlation_matrix, trajectory_summary,
    run_full_validation
)
from src.visualization import (
    plot_trajectory_lengths, plot_sensor_distributions,
    plot_sensor_correlation_heatmap, plot_sensor_trajectories,
    plot_operating_conditions, plot_degradation_examples,
    plot_sensor_cycle_correlation
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
%matplotlib inline

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\himan\OneDrive\Documents\AeroRUL


## 3. Dataset Discovery

In [5]:
# Discover which C-MAPSS subsets are available on disk
discovery = discover_datasets()

for subset, avail in discovery.items():
    meta = SUBSET_METADATA.get(subset, {})
    print(f"{subset}: train={avail['train']}, test={avail['test']}, rul={avail['rul']}")
    print(f"  -> {meta.get('description', 'N/A')}")

FD001: train=True, test=True, rul=True
  -> ONE condition (Sea Level), ONE fault mode (HPC Degradation)
FD002: train=True, test=True, rul=True
  -> SIX conditions, ONE fault mode (HPC Degradation)
FD003: train=True, test=True, rul=True
  -> ONE condition (Sea Level), TWO fault modes (HPC + Fan Degradation)
FD004: train=True, test=True, rul=True
  -> SIX conditions, TWO fault modes (HPC + Fan Degradation)


## 4. Loading C-MAPSS

In [6]:
# Load all available subsets
all_data = load_all_subsets()

for subset, data in all_data.items():
    print(f"{subset}: train={data['train'].shape}, test={data['test'].shape}, rul={'available' if data['rul'] is not None else 'missing'}")

# Quick look at FD001 training data
print("\n--- FD001 Train (first 5 rows) ---")
all_data['FD001']['train'].head()

FD001: train=(20631, 26), test=(13096, 26), rul=available
FD002: train=(53759, 26), test=(33991, 26), rul=available
FD003: train=(24720, 26), test=(16596, 26), rul=available
FD004: train=(61249, 26), test=(41214, 26), rul=available

--- FD001 Train (first 5 rows) ---


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## 5. Schema Validation

In [7]:
# Validate structure for all subsets
for subset, data in all_data.items():
    for split_name, df in [('train', data['train']), ('test', data['test'])]:
        result = validate_structure(df, f"{subset}_{split_name}")
        status = 'PASS' if result['column_count_ok'] and result['column_names_ok'] else 'FAIL'
        print(f"{subset} {split_name}: {result['n_rows']} rows x {result['n_cols']} cols | "
              f"Schema: {status} | Dup rows: {result['n_duplicate_rows']} | "
              f"Dup (unit,cycle): {result['n_duplicate_unit_cycle']}")

FD001 train: 20631 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD001 test: 13096 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD002 train: 53759 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD002 test: 33991 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD003 train: 24720 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD003 test: 16596 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD004 train: 61249 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0
FD004 test: 41214 rows x 26 cols | Schema: PASS | Dup rows: 0 | Dup (unit,cycle): 0


## 6. Data Quality Checks

In [8]:
# Missing value check for all subsets
for subset, data in all_data.items():
    for split_name, df in [('train', data['train']), ('test', data['test'])]:
        missing = validate_missing_values(df)
        total_missing = missing['n_total_missing'].sum()
        print(f"{subset} {split_name}: {total_missing} total missing/infinite values")
        if total_missing > 0:
            print(missing[missing['n_total_missing'] > 0])

FD001 train: 0 total missing/infinite values
FD001 test: 0 total missing/infinite values
FD002 train: 0 total missing/infinite values
FD002 test: 0 total missing/infinite values
FD003 train: 0 total missing/infinite values
FD003 test: 0 total missing/infinite values
FD004 train: 0 total missing/infinite values
FD004 test: 0 total missing/infinite values


## 7. Training Trajectory Analysis

In [9]:
# Training trajectory statistics
train_summaries = []
for subset, data in all_data.items():
    ts = trajectory_summary(data['train'], f"{subset}_train")
    train_summaries.append(ts)
    engines = validate_engines(data['train'])
    
    print(f"\n{'='*50}")
    print(f"{subset} TRAINING TRAJECTORIES")
    print(f"{'='*50}")
    print(f"  Engines:          {ts['n_engines']}")
    print(f"  Total obs:        {ts['n_observations']}")
    print(f"  Avg length:       {ts['avg_trajectory_length']}")
    print(f"  Min length:       {ts['min_trajectory_length']}")
    print(f"  Max length:       {ts['max_trajectory_length']}")
    print(f"  Median length:    {ts['median_trajectory_length']}")
    print(f"  Cycles ordered:   {engines['cycles_ordered']}")
    print(f"  Cycles start @1:  {engines['cycles_start_at_1']}")


FD001 TRAINING TRAJECTORIES
  Engines:          100
  Total obs:        20631
  Avg length:       206.31
  Min length:       128
  Max length:       362
  Median length:    199.0
  Cycles ordered:   True
  Cycles start @1:  True

FD002 TRAINING TRAJECTORIES
  Engines:          260
  Total obs:        53759
  Avg length:       206.77
  Min length:       128
  Max length:       378
  Median length:    199.0
  Cycles ordered:   True
  Cycles start @1:  True

FD003 TRAINING TRAJECTORIES
  Engines:          100
  Total obs:        24720
  Avg length:       247.2
  Min length:       145
  Max length:       525
  Median length:    220.5
  Cycles ordered:   True
  Cycles start @1:  True

FD004 TRAINING TRAJECTORIES
  Engines:          249
  Total obs:        61249
  Avg length:       245.98
  Min length:       128
  Max length:       543
  Median length:    234.0
  Cycles ordered:   True
  Cycles start @1:  True


In [10]:
# Engine final cycles (FD001 example)
final_cycles = get_engine_final_cycles(all_data['FD001']['train'])
print("FD001 — Engine Final Cycles (first 20 engines):")
final_cycles.head(20)

FD001 — Engine Final Cycles (first 20 engines):


,unit_id,final_cycle
0,1,192
1,2,287
2,3,179
3,4,189
4,5,269
5,6,188
6,7,259
7,8,150
8,9,201
9,10,222


## 8. Test Trajectory Analysis

In [11]:
# Test trajectory statistics
for subset, data in all_data.items():
    ts = trajectory_summary(data['test'], f"{subset}_test")
    engines = validate_engines(data['test'])
    
    print(f"\n{'='*50}")
    print(f"{subset} TEST TRAJECTORIES")
    print(f"{'='*50}")
    print(f"  Engines:          {ts['n_engines']}")
    print(f"  Total obs:        {ts['n_observations']}")
    print(f"  Avg length:       {ts['avg_trajectory_length']}")
    print(f"  Min length:       {ts['min_trajectory_length']}")
    print(f"  Max length:       {ts['max_trajectory_length']}")
    print(f"  Median length:    {ts['median_trajectory_length']}")
    
    # Inspect RUL file structure
    if data['rul'] is not None:
        rul = data['rul']
        print(f"  RUL entries:      {len(rul)}")
        print(f"  RUL range:        [{rul['rul'].min()}, {rul['rul'].max()}]")
        print(f"  RUL mean:         {rul['rul'].mean():.1f}")

print("\n--- Key distinction ---")
print("Training: engine observed until simulated failure (last cycle = failure)")
print("Test:     engine trajectory stops BEFORE failure (RUL > 0)")


FD001 TEST TRAJECTORIES
  Engines:          100
  Total obs:        13096
  Avg length:       130.96
  Min length:       31
  Max length:       303
  Median length:    133.5
  RUL entries:      100
  RUL range:        [7, 145]
  RUL mean:         75.5

FD002 TEST TRAJECTORIES
  Engines:          259
  Total obs:        33991
  Avg length:       131.24
  Min length:       21
  Max length:       367
  Median length:    132.0
  RUL entries:      259
  RUL range:        [6, 194]
  RUL mean:         81.2

FD003 TEST TRAJECTORIES
  Engines:          100
  Total obs:        16596
  Avg length:       165.96
  Min length:       38
  Max length:       475
  Median length:    148.0
  RUL entries:      100
  RUL range:        [6, 145]
  RUL mean:         75.3

FD004 TEST TRAJECTORIES
  Engines:          248
  Total obs:        41214
  Avg length:       166.19
  Min length:       19
  Max length:       486
  Median length:    153.5
  RUL entries:      248
  RUL range:        [6, 195]
  RUL mean:  

## 9. Operating Condition Analysis

In [12]:
# Operating condition analysis per subset
for subset, data in all_data.items():
    oc = analyze_operating_conditions(data['train'])
    print(f"\n{subset} Operating Conditions (Training):")
    print(oc.to_string(index=False))

print("\n--- Interpretation ---")
print("FD001/FD003: Single operating condition -> op_settings nearly constant")
print("FD002/FD004: Six operating conditions -> significant variation in op_settings")
print("This makes FD002/FD004 harder due to operating-regime confounding.")


FD001 Operating Conditions (Training):
     setting  n_unique       mean      std      min      max
op_setting_1       158  -0.000009 0.002187  -0.0087   0.0087
op_setting_2        13   0.000002 0.000293  -0.0006   0.0006
op_setting_3         1 100.000000 0.000000 100.0000 100.0000

FD002 Operating Conditions (Training):
     setting  n_unique      mean       std  min     max
op_setting_1       536 23.998407 14.747376  0.0  42.008
op_setting_2       105  0.572056  0.310016  0.0   0.842
op_setting_3         2 94.046020 14.237735 60.0 100.000

FD003 Operating Conditions (Training):
     setting  n_unique       mean      std      min      max
op_setting_1       160  -0.000024 0.002194  -0.0086   0.0086
op_setting_2        14   0.000005 0.000294  -0.0006   0.0007
op_setting_3         1 100.000000 0.000000 100.0000 100.0000

FD004 Operating Conditions (Training):
     setting  n_unique      mean       std  min     max
op_setting_1       536 23.999823 14.780722  0.0  42.008
op_setting_2    

## 10. Sensor Analysis

In [13]:
# Sensor statistics for FD001
fd001_sensors = validate_sensors(all_data['FD001']['train'])
print("FD001 — Sensor Statistics (Training):")
fd001_sensors

FD001 — Sensor Statistics (Training):


,sensor,min,max,mean,std,var,n_unique,pct_missing,is_constant,is_near_constant
0,sensor_1,518.6700,518.6700,518.670000,0.000000,0.000000,1,0.0,True,True
1,sensor_2,641.2100,644.5300,642.680934,0.500053,0.250053,310,0.0,False,True
2,sensor_3,1571.0400,1616.9100,1590.523119,6.131150,37.590994,3012,0.0,False,False
3,sensor_4,1382.2500,1441.4900,1408.933782,9.000605,81.010886,4051,0.0,False,False
4,sensor_5,14.6200,14.6200,14.620000,0.000000,0.000000,1,0.0,True,True
5,sensor_6,21.6000,21.6100,21.609803,0.001389,0.000002,2,0.0,False,True
6,sensor_7,549.8500,556.0600,553.367711,0.885092,0.783388,513,0.0,False,False
7,sensor_8,2387.9000,2388.5600,2388.096652,0.070985,0.005039,53,0.0,False,True
8,sensor_9,9021.7300,9244.5900,9065.242941,22.082880,487.653568,6403,0.0,False,False
9,sensor_10,1.3000,1.3000,1.300000,0.000000,0.000000,1,0.0,True,True


In [14]:
# Identify constant/near-constant sensors
for subset, data in all_data.items():
    sensors = validate_sensors(data['train'])
    constant = sensors[sensors['is_constant']]['sensor'].tolist()
    near_const = sensors[sensors['is_near_constant'] & ~sensors['is_constant']]['sensor'].tolist()
    print(f"{subset}: constant={constant}, near-constant={near_const}")

FD001: constant=['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19'], near-constant=['sensor_2', 'sensor_6', 'sensor_8', 'sensor_13']
FD002: constant=[], near-constant=[]
FD003: constant=['sensor_1', 'sensor_5', 'sensor_16', 'sensor_18', 'sensor_19'], near-constant=['sensor_2', 'sensor_6', 'sensor_8', 'sensor_13']
FD004: constant=[], near-constant=[]


In [15]:
# Sensor-cycle correlation (FD001 training)
corr = sensor_cycle_correlation(all_data['FD001']['train'])
print("FD001 — Sensor-Cycle Pearson Correlation:")
print(corr.sort_values('pearson_r', ascending=False).to_string(index=False))
print("\nNote: This is an initial exploratory statistic only.")
print("Sensor selection will be addressed in a later phase.")

FD001 — Sensor-Cycle Pearson Correlation:
   sensor  pearson_r
sensor_11   0.634385
 sensor_4   0.624577
sensor_15   0.588676
sensor_17   0.566995
 sensor_2   0.549898
 sensor_3   0.543947
sensor_13   0.477523
 sensor_8   0.475977
 sensor_9   0.443999
sensor_14   0.370324
 sensor_6   0.105980
sensor_16  -0.000000
 sensor_5  -0.000000
sensor_20  -0.583597
sensor_21  -0.585923
 sensor_7  -0.595914
sensor_12  -0.611354
 sensor_1        NaN
sensor_10        NaN
sensor_18        NaN
sensor_19        NaN

Note: This is an initial exploratory statistic only.
Sensor selection will be addressed in a later phase.


c:\Users\himan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\himan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 11. Visualizations

In [16]:
# 1. Trajectory length histograms
plot_trajectory_lengths(all_data)
print("Saved: trajectory_lengths.png")

Saved: trajectory_lengths.png


In [17]:
# 2. Sensor distributions (FD001)
plot_sensor_distributions(all_data['FD001']['train'], 'FD001')
print("Saved: sensor_distributions_FD001.png")

Saved: sensor_distributions_FD001.png


In [18]:
# 3. Sensor correlation heatmap (FD001)
corr_matrix = sensor_correlation_matrix(all_data['FD001']['train'])
plot_sensor_correlation_heatmap(corr_matrix, 'FD001')
print("Saved: sensor_correlation_FD001.png")

Saved: sensor_correlation_FD001.png


In [19]:
# 4. Sensor trajectories (FD001)
plot_sensor_trajectories(all_data['FD001']['train'], 'FD001')
print("Saved: sensor_trajectories_FD001.png")

Saved: sensor_trajectories_FD001.png


In [20]:
# 5. Operating condition distributions
plot_operating_conditions(all_data)
print("Saved: operating_conditions.png")

Saved: operating_conditions.png


In [21]:
# 6. Degradation examples (FD001)
plot_degradation_examples(all_data['FD001']['train'], 'FD001')
print("Saved: degradation_examples_FD001.png")

Saved: degradation_examples_FD001.png


In [22]:
# 7. Sensor-cycle correlation bar chart (FD001)
corr_df = sensor_cycle_correlation(all_data['FD001']['train'])
plot_sensor_cycle_correlation(corr_df, 'FD001')
print("Saved: sensor_cycle_correlation_FD001.png")

c:\Users\himan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\himan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Saved: sensor_cycle_correlation_FD001.png


## 12. Phase 1 Findings

### Key Observations

1. **All four C-MAPSS subsets (FD001-FD004) are available** with complete train/test/RUL files.

2. **Data quality is excellent**: No missing values, no duplicates, cycles are monotonically increasing and start at 1 for all engines across all subsets.

3. **Schema is consistent**: All files have exactly 26 columns (unit_id, cycle, 3 op_settings, 21 sensors).

4. **Operating conditions**:
   - FD001/FD003: Single condition (op_settings near-constant)
   - FD002/FD004: Six conditions (significant op_setting variation, confounding sensor readings)

5. **Constant/near-constant sensors**: Several sensors (e.g., sensor_1, sensor_5, sensor_10, sensor_16, sensor_18, sensor_19) appear constant or near-constant in FD001/FD003. These may need removal in later phases, but are preserved for now.

6. **Sensor degradation trends**: Many sensors show clear monotonic trends with cycle, indicating degradation signal.

### Recommended Next Phase

**Phase 2** should focus on:
- RUL target generation (remaining cycles for training data)
- RUL clipping experiments (e.g., max RUL = 125)
- Leakage-safe engine-level train/validation splitting
- Operating-condition normalization for FD002/FD004